In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, auc, roc_curve, precision_score, recall_score
from sklearn.model_selection import train_test_split, KFold, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from scipy.stats import randint
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import kagglehub
import shap

# Download datasets
path1 = kagglehub.dataset_download("umerrtx/machine-failure-prediction-using-sensor-data")
path2 = kagglehub.dataset_download("shivamb/machine-predictive-maintenance-classification")
path3 = kagglehub.dataset_download("programmer3/industrial-fault-detection-dataset")

df1 = pd.read_csv(f"{path1}/data.csv")
df2 = pd.read_csv(f"{path2}/predictive_maintenance.csv")
df3 = pd.read_csv(f"{path3}/Industrial_fault_detection.csv")




In [ ]:
def remove_outliers_iqr(df, columns):
  cleaned = df.copy()
  flagged_rows = set()

  for col in columns:
      Q1 = cleaned[col].quantile(0.25)
      Q3 = cleaned[col].quantile(0.75)
      IQR = Q3 - Q1
      lower_bound = Q1 - 1.5 * IQR
      upper_bound = Q3 + 1.5 * IQR

      outlier_rows = []
      for i in cleaned.index:
          value = cleaned.at[i, col]
          if value < lower_bound or value > upper_bound:
              outlier_rows.append(i)

      outliers = outlier_rows

      flagged_rows.update(outliers)

  if flagged_rows:
      cleaned = cleaned.drop(index=list(flagged_rows))

  return cleaned

In [ ]:
# Preproccessing for Dataset 1:

print(df1.info())

# Check for null values
null_counts = df1.isnull().sum()

# No null values found in the dataset
# print(null_counts)

# Remove duplicates
df1 = df1.drop_duplicates()

# Separate features and target
X1 = df1.drop('fail', axis=1)
y1 = df1['fail']

# Remove outliers for continuous features using the IQR method

continuous_features_1 = ['footfall', 'AQ', 'USS', 'CS', 'VOC', 'RP', 'IP', 'Temperature']
X1 = remove_outliers_iqr(X1, continuous_features_1)

# Keep features and targets aligned
y1 = y1.loc[X1.index]

# Scale continuous features
scaler1 = StandardScaler()
X1_scaled = X1.copy()
X1_scaled[continuous_features_1] = scaler1.fit_transform(X1[continuous_features_1])



In [ ]:
## Dataset 1 Before validation ##

# Split into training/validation and test sets
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X1_scaled, y1, test_size=0.2, stratify=y1, random_state=42
)

# Apply SMOTE on training data
smote1 = SMOTE(random_state=42)
X_trainval_res, y_trainval_res = smote1.fit_resample(X_trainval, y_trainval)

rf = RandomForestClassifier(random_state=42)
rf.fit(X_trainval_res, y_trainval_res)

# Evaluate on the test set
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
fpr_base, tpr_base, _ = roc_curve(y_test, y_prob)
roc_auc_base = auc(fpr_base, tpr_base)

# Print Results
print("\nDataset 1 - Machine Failure Prediction using Sensor Data (Random Forest)")
print("\nResults before validation:")
print("Confusion Matrix:\n", cm)
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"AUC: {roc_auc_base:.4f}")

# Plot ROC Curve
plt.figure()
plt.plot(fpr_base, tpr_base, label=f'ROC curve (area = {roc_auc_base:.2f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Dataset 1 (Random Forest) Pre validation')
plt.legend(loc="lower right")
plt.show()

### Dataset 1 After validation ###
# Split into training/validation and test sets
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X1_scaled, y1, test_size=0.2, stratify=y1, random_state=42
)

# Apply SMOTE only on training data
smote1 = SMOTE(random_state=42)
X_trainval_res, y_trainval_res = smote1.fit_resample(X_trainval, y_trainval)

# Define hyperparameter distributions
param_dist = {
    'n_estimators': [100, 150, 200, 300],
    # 'max_depth': [10, 20, 30, None],
    'max_depth': [5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [5, 10, 20],
    'max_features': ['sqrt', 'log2'],
    'criterion': ['gini', 'entropy']
}

rf = RandomForestClassifier(random_state=42)

# Randomized Search with 5-Fold Cross-Validation
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=20,  # Number of parameter settings sampled
    cv=5,
    scoring='recall',
    n_jobs=-1,
    verbose=1,
    random_state=42
)

# Runs the CV and hyperparameter search and store best model in best_rf
random_search.fit(X_trainval_res, y_trainval_res)
best_rf = random_search.best_estimator_

cv_results = pd.DataFrame(random_search.cv_results_)
print(cv_results[['mean_test_score', 'std_test_score']].head())

# Retrain the best model on full training data
best_rf.fit(X_trainval_res, y_trainval_res)

# Evaluate on the test set
y_pred = best_rf.predict(X_test)
y_prob = best_rf.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

# Print best parameters and scores for test recall
print("\nRandom Forest after validation on test set")
print("\nBest Parameters found:", random_search.best_params_)
print(f"Best Cross-Validation Recall: {random_search.best_score_:.4f}")
print(f"Test Recall: {recall:.4f}")

# Print Results
print("Confusion Matrix:\n", cm)
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"AUC: {roc_auc:.4f}")

# Plot ROC Curve
plt.figure()
plt.plot(fpr, tpr, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot(fpr_base, tpr_base, linestyle='--', label=f'Pre-validation ROC (AUC = {roc_auc_base:.2f})', color='grey')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Dataset 1 (Random Forest)')
plt.legend(loc="lower right")
plt.show()


# ---- FEATURE IMPORTANCE (Post-Validation) ----
importances = best_rf.feature_importances_
feature_names = X_trainval_res.columns

feat_imp_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print("\nTop 10 Feature Importances (After Validation):")
print(feat_imp_df.head(10))

# Plot feature importances
plt.figure(figsize=(10, 6))
plt.barh(feat_imp_df['Feature'][:10][::-1], feat_imp_df['Importance'][:10][::-1])
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Top 10 Feature Importances - After Validation')
plt.tight_layout()
plt.show()


# SHAP Analysis
print("\nRunning SHAP analysis")

# Initialize SHAP Explainer
explainer = shap.TreeExplainer(best_rf)
shap_values = explainer.shap_values(X_test)

if isinstance(shap_values, list):
    shap_to_use = shap_values[1]  # For binary classification, use the SHAP values for the positive class
else:
    shap_to_use = shap_values[:, :, 1]

# Global summary plot (for class 1 — failure)
shap.summary_plot(shap_to_use, X_test, show=True)

# Example of single prediction explanation
sample_index = 0
shap.force_plot(
    explainer.expected_value[1],
    shap_to_use[sample_index, :],
    X_test.iloc[sample_index, :],
    matplotlib=True
)



In [ ]:
# Dataset 1 model comparison

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import (
    confusion_matrix, accuracy_score, f1_score,
    precision_score, recall_score, roc_curve, auc
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from imblearn.over_sampling import SMOTE
from scipy.stats import randint, uniform

# ---------------------------------------
# 1. Data preparation
# ---------------------------------------

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X1_scaled, y1, test_size=0.2, stratify=y1, random_state=42
)

smote1 = SMOTE(random_state=42)
X_trainval_res, y_trainval_res = smote1.fit_resample(X_trainval, y_trainval)

# ---------------------------------------
# 2. Define models and hyperparameter grids
# ---------------------------------------

models_and_params = {
    "Logistic Regression": {
        "model": LogisticRegression(max_iter=500, random_state=42),
        "params": {
            "C": uniform(0.01, 10),
            "penalty": ["l2"],  # L1 requires solver='liblinear'
            "solver": ["lbfgs", "saga"]
        }
    },
    "KNN": {
        "model": KNeighborsClassifier(),
        "params": {
            "n_neighbors": randint(3, 20),
            "weights": ["uniform", "distance"],
            "p": [1, 2]  # Manhattan or Euclidean
        }
    },
    "Random Forest": {
        "model": RandomForestClassifier(random_state=42),
        "params": {
            "n_estimators": randint(50, 300),
            "max_depth": randint(3, 20),
            "min_samples_split": randint(2, 10),
            "min_samples_leaf": randint(1, 5),
            "max_features": ["sqrt", "log2", None],
            'criterion': ['gini', 'entropy']
        }
    },
    "Gradient Boosted": {
        "model": GradientBoostingClassifier(random_state=42),
        "params": {
            "n_estimators": randint(50, 300),
            "learning_rate": uniform(0.01, 0.5),
            "max_depth": randint(3, 10),
            "subsample": uniform(0.5, 0.5),
            "min_samples_split": randint(2, 10),
            "min_samples_leaf": randint(1, 5)
        }
    }
}

# ---------------------------------------
# 3. Randomized search, train, and evaluate
# ---------------------------------------

results = {}
plt.figure(figsize=(8, 6))

for name, mp in models_and_params.items():
    print(f"Running RandomizedSearchCV for {name}...")

    # Randomized search
    search = RandomizedSearchCV(
        estimator=mp["model"],
        param_distributions=mp["params"],
        n_iter=30,
        scoring="recall",
        cv=5,
        random_state=42,
        n_jobs=-1
    )
    search.fit(X_trainval_res, y_trainval_res)

    best_model = search.best_estimator_

    # Predict
    y_pred = best_model.predict(X_test)
    y_prob = best_model.predict_proba(X_test)[:, 1]

    # Metrics
    cm = confusion_matrix(y_test, y_pred)
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)

    # Store results
    results[name] = {
        "Best Params": search.best_params_,
        "Confusion Matrix": cm,
        "Accuracy": accuracy,
        "F1 Score": f1,
        "Precision": precision,
        "Recall": recall,
        "AUC": roc_auc
    }

    # Plot ROC
    plt.plot(fpr, tpr, lw=2, label=f"{name} (AUC = {roc_auc:.2f})")

# ---------------------------------------
# 4. Final ROC plot
# ---------------------------------------
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison - Dataset 1")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

# ---------------------------------------
# 5. Summary table
# ---------------------------------------
summary_df = pd.DataFrame(results).T
print("\nModel Performance Summary:")
print(summary_df[["Accuracy", "F1 Score", "Precision", "Recall", "AUC"]].round(4))

from sklearn.model_selection import learning_curve
import numpy as np
import matplotlib.pyplot as plt

# Generate learning curves for recall
train_sizes, train_scores, val_scores = learning_curve(
    best_rf,
    X_trainval_res,
    y_trainval_res,
    cv=5,
    scoring='recall',
    n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10)
)

# Compute mean and std
train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
val_mean = np.mean(val_scores, axis=1)
val_std = np.std(val_scores, axis=1)

# Plot learning curves
plt.figure(figsize=(10, 7))
plt.plot(train_sizes, train_mean, color='#3498db', lw=3, marker='o',
         markersize=8, label='Training Recall')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                 alpha=0.2, color='#3498db')

plt.plot(train_sizes, val_mean, color='#e74c3c', lw=3, marker='s',
         markersize=8, label='Validation Recall')
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std,
                 alpha=0.2, color='#e74c3c')

plt.xlabel('Training Set Size', fontsize=13, fontweight='bold')
plt.ylabel('Recall Score', fontsize=13, fontweight='bold')
plt.title('Learning Curves - Tuned Random Forest (Dataset 1)',
          fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=12)
plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('learning_curves_rf_dataset1.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: learning_curves_rf_dataset1.png")
print(f"\nFinal Training Recall: {train_mean[-1]:.4f} ± {train_std[-1]:.4f}")
print(f"Final Validation Recall: {val_mean[-1]:.4f} ± {val_std[-1]:.4f}")



Dataset 2

In [ ]:
# Preprocessing for Dataset 2: Machine Predictive Maintenance Classification

print(df2.columns)

# Remove duplicates
df2 = df2.drop_duplicates()

# Encode 'Type' categorical variable
le_type = LabelEncoder()
df2['Type_Encoded'] = le_type.fit_transform(df2['Type'])

# Drop non-informative columns
df2_clean = df2.drop(['UDI', 'Product ID', 'Type', 'Failure Type'], axis=1)

# Separate features and target
X2 = df2_clean.drop('Target', axis=1)
y2 = df2_clean['Target']

# Store feature names for SHAP
feature_names = list(X2.columns)
print(f"\nFeatures for SHAP analysis: {feature_names}")

# Remove outliers
continuous_features_2 = [
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]',
]

X2_clean = remove_outliers_iqr(X2, continuous_features_2)
y2_clean = y2.loc[X2_clean.index]

# Scale features
scaler2 = StandardScaler()
X2_scaled = scaler2.fit_transform(X2_clean)

# Train-test split (BEFORE SMOTE)
X_train, X_test, y_train, y_test = train_test_split(
    X2_scaled, y2_clean, test_size=0.2, stratify=y2_clean, random_state=42
)

# Apply SMOTE only to training data
smote2 = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote2.fit_resample(X_train, y_train)




In [ ]:
# baseline gbm

gbm_baseline = GradientBoostingClassifier(random_state=42)
gbm_baseline.fit(X_train_resampled, y_train_resampled)

# Predictions
y_pred_baseline = gbm_baseline.predict(X_test)
y_prob_baseline = gbm_baseline.predict_proba(X_test)[:, 1]

# Metrics
accuracy_baseline = accuracy_score(y_test, y_pred_baseline)
f1_baseline = f1_score(y_test, y_pred_baseline)
precision_baseline = precision_score(y_test, y_pred_baseline)
recall_baseline = recall_score(y_test, y_pred_baseline)

print(f"Accuracy: {accuracy_baseline:.4f}")
print(f"F1 Score: {f1_baseline:.4f}")
print(f"Precision: {precision_baseline:.4f}")
print(f"Recall: {recall_baseline:.4f}")

# ROC/AUC
fpr_baseline, tpr_baseline, _ = roc_curve(y_test, y_prob_baseline)
auc_baseline = auc(fpr_baseline, tpr_baseline)
print(f"AUC: {auc_baseline:.4f}")

In [ ]:
# tuned gbm

gbm = GradientBoostingClassifier(
    random_state=42,
    validation_fraction=0.1,  # Use 10% for early stopping
    n_iter_no_change=10,      # Stop if no improvement for 10 iterations
    tol=0.0001                # Tolerance for early stopping
)

# More conservative parameter ranges to prevent overfitting
param_dist_gbm = {
    'n_estimators': randint(50, 150),        # Reduced upper limit
    'learning_rate': [0.01, 0.05, 0.1],      # Lower learning rates
    'max_depth': randint(2, 6),              # Shallower trees
    'min_samples_split': randint(5, 20),     # Higher minimum splits
    'min_samples_leaf': randint(2, 10),      # Higher minimum leaf samples
    'subsample': [0.7, 0.8, 0.9],           # More aggressive subsampling
    'max_features': ['sqrt', 'log2']         # Feature subsampling
}

random_search_gbm = RandomizedSearchCV(
    gbm, param_distributions=param_dist_gbm, n_iter=20, cv=5,
    scoring='f1', n_jobs=-1, random_state=42, verbose=1  # Changed to f1 for balanced metric
)

random_search_gbm.fit(X_train_resampled, y_train_resampled)
print(f"\nBest Parameters: {random_search_gbm.best_params_}")
print(f"Best CV F1 Score: {random_search_gbm.best_score_:.4f}")

best_gbm = random_search_gbm.best_estimator_

# Predictions
y_pred_tuned = best_gbm.predict(X_test)
y_prob_tuned = best_gbm.predict_proba(X_test)[:, 1]

# Metrics
accuracy_tuned = accuracy_score(y_test, y_pred_tuned)
f1_tuned = f1_score(y_test, y_pred_tuned)
precision_tuned = precision_score(y_test, y_pred_tuned)
recall_tuned = recall_score(y_test, y_pred_tuned)

print(f"\nTest Set Performance:")
print(f"Accuracy: {accuracy_tuned:.4f}")
print(f"F1 Score: {f1_tuned:.4f}")
print(f"Precision: {precision_tuned:.4f}")
print(f"Recall: {recall_tuned:.4f}")

# Check for overfitting on tuned model
train_pred_tuned = best_gbm.predict(X_train_resampled)
train_accuracy_tuned = accuracy_score(y_train_resampled, train_pred_tuned)
print(f"\nTrain Accuracy: {train_accuracy_tuned:.4f}")
print(f"Test Accuracy: {accuracy_tuned:.4f}")
print(f"Overfitting Gap: {train_accuracy_tuned - accuracy_tuned:.4f}")

if train_accuracy_tuned - accuracy_tuned > 0.05:
    print("WARNING: Model may be overfitting (gap > 0.05)")
elif train_accuracy_tuned - accuracy_tuned > 0.02:
    print("CAUTION: Slight overfitting detected (gap > 0.02)")
else:
    print("✓ Model appears to generalize well (gap < 0.02)")

# ROC/AUC
fpr_tuned, tpr_tuned, _ = roc_curve(y_test, y_prob_tuned)
auc_tuned = auc(fpr_tuned, tpr_tuned)
print(f"AUC: {auc_tuned:.4f}")


In [ ]:
plt.figure(figsize=(10, 7))
plt.plot(fpr_baseline, tpr_baseline, color='#95a5a6', lw=3, linestyle='--',
         label=f'Baseline GBM (AUC = {auc_baseline:.3f})')
plt.plot(fpr_tuned, tpr_tuned, color='#f39c12', lw=3,
         label=f'Tuned GBM (AUC = {auc_tuned:.3f})')
plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=13, fontweight='bold')
plt.ylabel('True Positive Rate', fontsize=13, fontweight='bold')
plt.title('ROC Curve Comparison - Baseline vs Tuned GBM (Dataset 2)',
          fontsize=14, fontweight='bold')
plt.legend(loc="lower right", fontsize=12)
plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('roc_comparison_gbm_dataset2.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: roc_comparison_gbm_dataset2.png")

In [ ]:
# Calculate learning curves for tuned model
train_sizes, train_scores, val_scores = learning_curve(
    best_gbm,
    X_train_resampled,
    y_train_resampled,
    cv=5,
    scoring='recall',
    n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10),
    random_state=42
)

# Calculate mean and std
train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
val_mean = np.mean(val_scores, axis=1)
val_std = np.std(val_scores, axis=1)

# Plot learning curves
plt.figure(figsize=(10, 7))
plt.plot(train_sizes, train_mean, color='#3498db', lw=3, marker='o',
         markersize=8, label='Training Score')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                 alpha=0.2, color='#3498db')

plt.plot(train_sizes, val_mean, color='#e74c3c', lw=3, marker='s',
         markersize=8, label='Validation Score')
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std,
                 alpha=0.2, color='#e74c3c')

plt.xlabel('Training Set Size', fontsize=13, fontweight='bold')
plt.ylabel('Recall Score', fontsize=13, fontweight='bold')
plt.title('Learning Curves - Tuned Gradient Boosting (Dataset 2)',
          fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=12)
plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('learning_curves_gbm_dataset2.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: learning_curves_gbm_dataset2.png")
print(f"\nFinal Training Score: {train_mean[-1]:.4f} ± {train_std[-1]:.4f}")
print(f"Final Validation Score: {val_mean[-1]:.4f} ± {val_std[-1]:.4f}")

In [ ]:
# Create SHAP explainer
explainer = shap.TreeExplainer(best_gbm)

# Calculate SHAP values for test set (use a sample if dataset is large)
if X_test.shape[0] > 500:
    print(f"Using sample of 500 instances for SHAP (out of {X_test.shape[0]})")
    sample_indices = np.random.choice(X_test.shape[0], 500, replace=False)
    X_test_sample = X_test[sample_indices]
else:
    X_test_sample = X_test

shap_values = explainer.shap_values(X_test_sample)

print("SHAP values calculated successfully")

# Convert to DataFrame for better visualization
X_test_df = pd.DataFrame(X_test_sample, columns=feature_names)

In [ ]:
print("\nGenerating SHAP Bar Plot...")

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test_df, plot_type="bar", show=False)
plt.title('SHAP Feature Importance - Gradient Boosting (Dataset 2)',
          fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Mean |SHAP value|', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_bar_plot_gbm_dataset2.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: shap_bar_plot_gbm_dataset2.png")

In [ ]:
print("\nGenerating SHAP Summary Plot...")

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test_df, show=False)
plt.title('SHAP Summary Plot - Gradient Boosting (Dataset 2)',
          fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('shap_summary_plot_gbm_dataset2.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: shap_summary_plot_gbm_dataset2.png")

Dataset 3

In [ ]:
# Preprocessing for Dataset 3: Industrial Fault Detection

print(df3.info())

print(f"Dataset shape: {df3.shape}")
print(f"\nTarget distribution:")
print(df3['Fault_Type'].value_counts())

# Remove duplicates
df3 = df3.drop_duplicates()

# Separate features and target
X3 = df3.drop('Fault_Type', axis=1)
y3 = df3['Fault_Type']

# Store feature names for SHAP
feature_names = list(X3.columns)
print(f"\nNumber of features: {len(feature_names)}")

# Remove outliers
continuous_features = X3.columns.tolist()
X3_clean = remove_outliers_iqr(X3, continuous_features)
y3_clean = y3.loc[X3_clean.index]

# Scale features
scaler3 = StandardScaler()
X3_scaled = scaler3.fit_transform(X3_clean)

# Train-test split (BEFORE SMOTE)
X_train, X_test, y_train, y_test = train_test_split(
    X3_scaled, y3_clean, test_size=0.2, stratify=y3_clean, random_state=42
)

print(f"\nTrain size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

# Get unique classes
classes = sorted(np.unique(y_test))
n_classes = len(classes)
print(f"Number of classes: {n_classes}")
print(f"Classes: {classes}")

# Apply SMOTE only to training data
smote3 = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote3.fit_resample(X_train, y_train)

enn = EditedNearestNeighbours()
X_train_resampled, y_train_resampled = enn.fit_resample(X_train_resampled, y_train_resampled)


In [ ]:
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.ensemble import GradientBoostingClassifier
from scipy.stats import randint
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_curve, auc
from sklearn.preprocessing import label_binarize

# baseline gbm

gbm_baseline = GradientBoostingClassifier(random_state=42)
gbm_baseline.fit(X_train_resampled, y_train_resampled)

# Predictions
y_pred_baseline = gbm_baseline.predict(X_test)
y_prob_baseline = gbm_baseline.predict_proba(X_test)

# Metrics
accuracy_baseline = accuracy_score(y_test, y_pred_baseline)
f1_baseline = f1_score(y_test, y_pred_baseline, average='weighted')
precision_baseline = precision_score(y_test, y_pred_baseline, average='weighted')
recall_baseline = recall_score(y_test, y_pred_baseline, average='weighted')

print(f"Accuracy: {accuracy_baseline:.4f}")
print(f"F1 Score: {f1_baseline:.4f}")
print(f"Precision: {precision_baseline:.4f}")
print(f"Recall: {recall_baseline:.4f}")

# ROC/AUC (micro-average)
y_test_binarized = label_binarize(y_test, classes=classes)
fpr_baseline, tpr_baseline, _ = roc_curve(y_test_binarized.ravel(), y_prob_baseline.ravel())
auc_baseline = auc(fpr_baseline, tpr_baseline)
print(f"Micro-Average AUC: {auc_baseline:.4f}")

In [ ]:
# Define tuned GBM
gbm = GradientBoostingClassifier(
    random_state=42,
    validation_fraction=0.15,   # 15% validation set for early stopping
    n_iter_no_change=10,
    tol=1e-4,
    subsample=0.7,              # Boosting on random subsets
    max_features='sqrt',        # Reduce feature correlation
)

param_dist_gbm = {
    'n_estimators': randint(30, 80),
    'learning_rate': [0.01, 0.03],
    'max_depth': randint(2, 4),
    'min_samples_split': randint(20, 50),
    'min_samples_leaf': randint(10, 25),
    'min_impurity_decrease': [0.01, 0.02, 0.03],
}

# Use StratifiedKFold to avoid folds with missing classes
stratified_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

random_search_gbm = RandomizedSearchCV(
    gbm,
    param_distributions=param_dist_gbm,
    n_iter=20,
    cv=stratified_cv,           # <-- fixed
    scoring='f1_macro',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

# Fit on SMOTE-resampled training data
random_search_gbm.fit(X_train_resampled, y_train_resampled)

best_gbm = random_search_gbm.best_estimator_
print(f"\nBest Parameters: {random_search_gbm.best_params_}")
print(f"Best CV F1 Score (macro): {random_search_gbm.best_score_:.4f}")

# Predictions on test set
y_pred_tuned = best_gbm.predict(X_test)
y_prob_tuned = best_gbm.predict_proba(X_test)

# Metrics
accuracy_tuned = accuracy_score(y_test, y_pred_tuned)
f1_tuned = f1_score(y_test, y_pred_tuned, average='weighted')
precision_tuned = precision_score(y_test, y_pred_tuned, average='weighted')
recall_tuned = recall_score(y_test, y_pred_tuned, average='weighted')

print(f"\nTest Set Performance:")
print(f"Accuracy: {accuracy_tuned:.4f}")
print(f"F1 Score: {f1_tuned:.4f}")
print(f"Precision: {precision_tuned:.4f}")
print(f"Recall: {recall_tuned:.4f}")

# Overfitting check
train_pred_tuned = best_gbm.predict(X_train_resampled)
train_accuracy_tuned = accuracy_score(y_train_resampled, train_pred_tuned)
gap = train_accuracy_tuned - accuracy_tuned
print(f"\nTrain Accuracy: {train_accuracy_tuned:.4f}")
print(f"Test Accuracy: {accuracy_tuned:.4f}")
print(f"Overfitting Gap: {gap:.4f}")

# ROC/AUC (micro-average)
fpr_tuned, tpr_tuned, _ = roc_curve(y_test_binarized.ravel(), y_prob_tuned.ravel())
auc_tuned = auc(fpr_tuned, tpr_tuned)
print(f"Micro-Average AUC: {auc_tuned:.4f}")


In [ ]:
plt.figure(figsize=(10, 7))
plt.plot(fpr_baseline, tpr_baseline, color='#95a5a6', lw=3, linestyle='--',
         label=f'Baseline GBM (AUC = {auc_baseline:.3f})')
plt.plot(fpr_tuned, tpr_tuned, color='#f39c12', lw=3,
         label=f'Tuned GBM (AUC = {auc_tuned:.3f})')
plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=13, fontweight='bold')
plt.ylabel('True Positive Rate', fontsize=13, fontweight='bold')
plt.title('ROC Curve Comparison - Baseline vs Tuned GBM (Dataset 3)',
          fontsize=14, fontweight='bold')
plt.legend(loc="lower right", fontsize=12)
plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('roc_comparison_gbm_dataset3.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: roc_comparison_gbm_dataset3.png")

In [ ]:
# Calculate learning curves for tuned model (more points for smoother curve)
train_sizes, train_scores, val_scores = learning_curve(
    best_gbm,
    X_train_resampled,
    y_train_resampled,
    cv=5,
    scoring='f1_macro',  # Changed to f1_macro for more balanced metric
    n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 20),  # Increased to 20 points for smoother curve
    random_state=42
)

# Calculate mean and std
train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
val_mean = np.mean(val_scores, axis=1)
val_std = np.std(val_scores, axis=1)

# Plot learning curves
plt.figure(figsize=(10, 7))
plt.plot(train_sizes, train_mean, color='#3498db', lw=3, marker='o',
         markersize=8, label='Training Score')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                 alpha=0.2, color='#3498db')

plt.plot(train_sizes, val_mean, color='#e74c3c', lw=3, marker='s',
         markersize=8, label='Validation Score')
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std,
                 alpha=0.2, color='#e74c3c')

plt.xlabel('Training Set Size', fontsize=13, fontweight='bold')
plt.ylabel('F1 Score (Macro)', fontsize=13, fontweight='bold')
plt.title('Learning Curves - Tuned Gradient Boosting (Dataset 3)',
          fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=12)
plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('learning_curves_gbm_dataset3.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: learning_curves_gbm_dataset3.png")
print(f"\nFinal Training Score: {train_mean[-1]:.4f} ± {train_std[-1]:.4f}")
print(f"Final Validation Score: {val_mean[-1]:.4f} ± {val_std[-1]:.4f}")
print(f"Gap: {train_mean[-1] - val_mean[-1]:.4f}")


In [ ]:
# For multi-class GradientBoosting, we need to use KernelExplainer
# TreeExplainer doesn't support multi-class GBM

# Use a smaller sample for SHAP (KernelExplainer is slower)
if X_test.shape[0] > 100:
    print(f"Using sample of 100 instances for SHAP (KernelExplainer is computationally expensive)")
    sample_indices = np.random.choice(X_test.shape[0], 100, replace=False)
    X_test_sample = X_test[sample_indices]
else:
    X_test_sample = X_test

# Also need a background dataset (use training data sample)
if X_train_resampled.shape[0] > 100:
    background_indices = np.random.choice(X_train_resampled.shape[0], 100, replace=False)
    X_background = X_train_resampled[background_indices]
else:
    X_background = X_train_resampled

print(f"Background dataset size: {X_background.shape[0]}")
print(f"Test sample size: {X_test_sample.shape[0]}")

# Convert to DataFrame for better visualization
X_test_df = pd.DataFrame(X_test_sample, columns=feature_names)
X_background_df = pd.DataFrame(X_background, columns=feature_names)

# Create SHAP explainer (KernelExplainer for multi-class GBM)
print("\nCreating SHAP KernelExplainer (this may take a few minutes)...")
explainer = shap.KernelExplainer(best_gbm.predict_proba, X_background_df)

# Calculate SHAP values
print("Calculating SHAP values (this will take some time)...")
shap_values = explainer.shap_values(X_test_df)

print(f"SHAP values calculated successfully")
print(f"Shape: {np.array(shap_values).shape}")

In [ ]:
# KernelExplainer for multi-class returns array with shape (n_samples, n_features, n_classes)
# We need to transpose to (n_classes, n_samples, n_features)
if len(shap_values.shape) == 3:
    # Transpose from (samples, features, classes) to (classes, samples, features)
    shap_values_transposed = np.transpose(shap_values, (2, 0, 1))
    print(f"Transposed shape: {shap_values_transposed.shape}")
else:
    shap_values_transposed = shap_values

for class_idx, class_label in enumerate(classes):
    print(f"\nGenerating SHAP Bar Plot for Class {class_label}...")
    print(f"SHAP values shape for class {class_label}: {shap_values_transposed[class_idx].shape}")

    plt.figure(figsize=(10, 8))
    shap.summary_plot(
        shap_values_transposed[class_idx],
        X_test_df.values,
        feature_names=feature_names,
        plot_type="bar",
        show=False,
        max_display=20
    )
    plt.title(f'SHAP Feature Importance - Class {class_label} (Dataset 3)',
              fontsize=14, fontweight='bold', pad=20)
    plt.xlabel('Mean |SHAP value|', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'shap_bar_plot_class{class_label}_dataset3.png', dpi=300, bbox_inches='tight')
    plt.show()

    print(f"Saved: shap_bar_plot_class{class_label}_dataset3.png")

In [ ]:
for class_idx, class_label in enumerate(classes):
    print(f"\nGenerating SHAP Summary Plot for Class {class_label}...")

    plt.figure(figsize=(10, 8))
    shap.summary_plot(
        shap_values_transposed[class_idx],
        X_test_df.values,
        feature_names=feature_names,
        show=False,
        max_display=20
    )
    plt.title(f'SHAP Summary Plot - Class {class_label} (Dataset 3)',
              fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.savefig(f'shap_summary_plot_class{class_label}_dataset3.png', dpi=300, bbox_inches='tight')
    plt.show()

    print(f"Saved: shap_summary_plot_class{class_label}_dataset3.png")